# Task 3 — Kafka Topic Design

## Mục tiêu

Tách node, edge, metadata và parser error thành bốn topic; mọi event mang schema
version và event time.

## Thiết kế và triển khai

Node, edge và metadata dùng cleanup `compact`, record key lần lượt là
`node_id`, `edge_id`, `file_id`. Error dùng cleanup `delete` với retention bảy
ngày. Contract và schema nằm tại
[`task3/TOPIC_CONTRACT.md`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task3/TOPIC_CONTRACT.md) và
[`task3/schemas/`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task3/schemas).

## Lệnh thực thi

```bash
bash task3/create_topics.sh
bash task3/list_topics.sh
bash task3/describe_topics.sh
bash task3/send_samples.sh
bash task3/consume_samples.sh
```

## Kết quả thực nghiệm


In [1]:
from pathlib import Path

topics = Path("../artifacts/task3/topics_list.txt").read_text(encoding="utf-8").split()
expected = {"cpg.nodes", "cpg.edges", "cpg.metadata", "cpg.errors"}
assert set(topics) == expected
print("topics:", ", ".join(sorted(topics)))

description = Path("../artifacts/task3/topics_describe.txt").read_text(encoding="utf-8")
for topic in sorted(expected):
    assert f"Topic: {topic}" in description
for line in description.splitlines():
    if "PartitionCount:" in line:
        print(line.strip())


topics: cpg.edges, cpg.errors, cpg.metadata, cpg.nodes
Topic: cpg.nodes	TopicId: dce7RYlHRVG5lJ4JkPqw4g	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=compact
Topic: cpg.edges	TopicId: HJDET6eTTvyROBqqG6d08w	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=compact
Topic: cpg.metadata	TopicId: Wn79q10fTq6RDRq0pk5whQ	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=compact
Topic: cpg.errors	TopicId: P92U0eFpScKOgibwRUwiJw	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=delete,retention.ms=604800000


## Vấn đề và cách xử lý

Script shell chạy trên Windows có nguy cơ CRLF và Compose từng nội suy biến
shell quá sớm. `.gitattributes` ép LF cho `*.sh`, còn biến chạy trong container
được escape để `docker compose config` giữ nguyên.

## Đánh giá

Bốn topic tồn tại với đúng cleanup policy và một partition cho môi trường lab.
Một partition đơn giản hóa thứ tự nhưng không đại diện cho cấu hình scale-out;
khi tăng partition phải giữ record key ổn định để cùng ID vào cùng partition.
